This notebook takes the icdar training data and generates a csv file with writer,same_text,isEng,train,file_name,male columns (file_name is the absolute path)

In [1]:
#imports
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import random
import sys
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

In [2]:
#test functions
def male_counts(sex_df):
    # Get the counts of each unique value in the "male" column
    male_counts = sex_df['male'].value_counts(dropna=False)

    # Print the counts
    print("Number of times 'male' is 0:", male_counts.get(0, 0))
    print("Number of times 'male' is 1:", male_counts.get(1, 0))
    print("Number of times 'male' is something else:", len(sex_df) - male_counts.get(0, 0) - male_counts.get(1, 0))
def check_if_both(train_df, column_name='same_text'):
    # Group by writer and check if both same_text=1 and same_text=0 are present
    writer_groups = train_df.groupby('writer')[column_name].nunique()

    # Filter writers that do not have both same_text=1 and same_text=0
    writers_missing_both = writer_groups[writer_groups != 2]

    if writers_missing_both.empty:
        print(f"All writers have both {column_name}=1 and {column_name}=0.")
    else:
        print(f"The following writers do not have {column_name}=1 and {column_name}=0")
        print(writers_missing_both)
def check_randomization(train_df):
    # Get the number of rows where train == 1
    train_1_count = train_df[train_df['train'] == 1].shape[0]

    # Calculate the fraction
    train_1_fraction = train_1_count / train_df.shape[0]

    print(f"Number of rows where train == 1: {train_1_count}")
    print(f"Fraction of rows where train == 1: {train_1_fraction:.2f}")
def check_grouping(train_df):
    # Group by writer and check if the train column has a constant value
    constant_train_check = train_df.groupby('writer')['train'].nunique()

    # Find writers where the train column is not constant
    non_constant_writers = constant_train_check[constant_train_check > 1]

    if non_constant_writers.empty:
        print("The train column is constant for all writers.")
    else:
        print("The train column is not constant for the following writers:")
        print(non_constant_writers)
def check_occurrences(train_df):
    # Count the occurrences of each unique writer value
    writer_counts = train_df['writer'].value_counts()

    # Check if all writers have exactly 4 occurrences
    if (writer_counts == 4).all():
        print("Each unique writer value occurs on exactly 4 rows.")
    else:
        print("Some writers do not occur exactly 4 times.")
        print(writer_counts[writer_counts != 4])
def check_title_association(train_df):
    random_numbers = random.sample(range(1, 282*4+1), 10)
    for n in random_numbers:
        print(n)
        print(train_df['file_name'][n])
        print(train_df['writer'][n],train_df['isEng'][n], train_df['same_text'][n])
        print('-------------')
def check_sex_association(train_df,sex_df):
    random_numbers = random.sample(range(1, 283), 10)
    for n in random_numbers:
        print(n)
        print(train_df[train_df['writer'] == n][['writer','male']])
        print(sex_df[sex_df['writer'] == n][['writer','male']])
        print('-------------')
def check_if_seed(train_df):
    train_0_writers = train_df[train_df['train'] == 0]['writer'].unique().tolist()
    train_1_writers = train_df[train_df['train'] == 1]['writer'].unique().tolist()
    return train_0_writers, train_1_writers


In [3]:
# Set the random seed for reproducibility
seed=42
np.random.seed(seed)

In [4]:
data_PATH="C:\\Users\\andre\\PhD\\Datasets\\ICDAR 2013 - Gender Identification Competition Dataset"
image_PATH=data_PATH+"\\unzipped"

In [5]:
sex_df = pd.read_csv(os.path.join(data_PATH, "test_answers.csv"),delimiter=',')
sex_df.head(15)

,writer,male,Usage
0,283,1,PrivateTest
1,284,0,PublicTest
2,285,0,PrivateTest
3,286,0,PublicTest
4,287,1,PublicTest
5,288,0,PublicTest
6,289,1,PrivateTest
7,290,1,PrivateTest
8,291,0,PrivateTest
9,292,0,PublicTest


In [6]:
private=sex_df[sex_df['Usage']=='PrivateTest']
public=sex_df[sex_df['Usage']=='PublicTest']
len(sex_df[sex_df['Usage']=='PrivateTest']) # 282
len(sex_df[sex_df['Usage']=='PublicTest']) # 282

71

In [7]:
male_counts(sex_df)
male_counts(public)

Number of times 'male' is 0: 111
Number of times 'male' is 1: 82
Number of times 'male' is something else: 0
Number of times 'male' is 0: 43
Number of times 'male' is 1: 28
Number of times 'male' is something else: 0


In [9]:
def is_valid_folder(folder):
    parts = folder.split('_')
    if len(parts) == 0:
        return False
    try:
        int(parts[0])
        return True
    except ValueError:
        return False
folder_names = [folder for folder in os.listdir(image_PATH) if os.path.isdir(os.path.join(image_PATH, folder)) and is_valid_folder(folder)]
# Extract the X part from the folder names
x_values = [int(folder.split('_')[0]) for folder in folder_names]

# Sort both lists based on the X values
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values[k])
folder_names = [folder_names[i] for i in sorted_indices]
x_values = [x_values[i] for i in sorted_indices]
print(folder_names)

['1_50', '51_100', '101_150', '151_200', '201_250', '251_300', '301_350', '351_400', '401_450', '451_475']


In [10]:
# Loop through each directory and collect image file paths for labeled images only
image_dirs = [os.path.join(image_PATH, folder) for folder in folder_names]
writers = []
isEng = []
same_text = []
file_names = []

for image_dir in image_dirs:
    for f in os.listdir(image_dir):
        if f.endswith('.jpg'):
            base_name = os.path.splitext(f)[0]  # Remove extension
            parts = base_name.split('_')

            if len(parts) != 2:
                continue  # Skip files that don't follow the expected pattern

            index, version = parts

            if int(version)>2:
                isEng.append(1)
            else:
                isEng.append(0)
            if int(version)%2==0:
                same_text.append(1)
            else:
                same_text.append(0)
            file_names.append(os.path.join(image_dir,f))
            writers.append(int(index))

# Create a dataframe from the extracted index and version values
train_file_df = pd.DataFrame({'writer': writers, 'isEng': isEng, 'same_text': same_text,'file_name':file_names})

# Display the dataframe
print(train_file_df['writer'].nunique())

check_if_both(train_file_df,column_name='same_text')
check_if_both(train_file_df, column_name='isEng')

475
All writers have both same_text=1 and same_text=0.
All writers have both isEng=1 and isEng=0.


In [11]:
N=282
test_df = train_file_df[train_file_df['writer']>N]
test_df = test_df.merge(sex_df, on=['writer'], how='left')
# Display the updated dataframe
test_df.head(10)

,writer,isEng,same_text,file_name,male,Usage
0,283,0,0,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,1,PrivateTest
1,283,0,1,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,1,PrivateTest
2,283,1,0,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,1,PrivateTest
3,283,1,1,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,1,PrivateTest
4,284,0,0,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,0,PublicTest
5,284,0,1,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,0,PublicTest
6,284,1,0,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,0,PublicTest
7,284,1,1,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,0,PublicTest
8,285,0,0,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,0,PrivateTest
9,285,0,1,C:\Users\andre\PhD\Datasets\ICDAR 2013 - Gende...,0,PrivateTest


In [12]:
check_sex_association(test_df,sex_df)

140
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
15
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
35
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
142
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
251
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
138
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
7
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
204
Empty DataFrame
Columns: [writer, male]
Index: []
Empty DataFrame
Columns: [writer, male]
Index: []
-------------
153
Empty DataFrame
Columns: [writer, male]
Index: []
Empty 

In [ ]:
check_title_association(test_df)

239
D:\download\PD project\datasets\ICDAR 2013 - Gender Identification Competition Dataset\unzipped\51_100\0060_4.jpg
60 1 1
-------------
951
D:\download\PD project\datasets\ICDAR 2013 - Gender Identification Competition Dataset\unzipped\201_250\0238_4.jpg
238 1 1
-------------
141
D:\download\PD project\datasets\ICDAR 2013 - Gender Identification Competition Dataset\unzipped\1_50\0036_2.jpg
36 0 1
-------------
1023
D:\download\PD project\datasets\ICDAR 2013 - Gender Identification Competition Dataset\unzipped\251_300\0256_4.jpg
256 1 1
-------------
883
D:\download\PD project\datasets\ICDAR 2013 - Gender Identification Competition Dataset\unzipped\201_250\0221_4.jpg
221 1 1
-------------
956
D:\download\PD project\datasets\ICDAR 2013 - Gender Identification Competition Dataset\unzipped\201_250\0240_1.jpg
240 0 0
-------------
731
D:\download\PD project\datasets\ICDAR 2013 - Gender Identification Competition Dataset\unzipped\151_200\0183_4.jpg
183 1 1
-------------
444
D:\download\PD

In [13]:
male_counts(test_df)
check_if_both(test_df, column_name='same_text')
check_if_both(test_df, column_name='isEng') 
check_randomization(test_df)
check_grouping(test_df)
check_occurrences(test_df)


Number of times 'male' is 0: 444
Number of times 'male' is 1: 328
Number of times 'male' is something else: 0
All writers have both same_text=1 and same_text=0.
All writers have both isEng=1 and isEng=0.


KeyError: 'train'

In [14]:
import json

def get_base_metadata(filepath):
    stats = os.stat(filepath)
    return {
        "full_path": os.path.abspath(filepath),
        "size_bytes": stats.st_size,
        "created": datetime.fromtimestamp(stats.st_ctime).isoformat(),
        "modified": datetime.fromtimestamp(stats.st_mtime).isoformat(),
        "accessed": datetime.fromtimestamp(stats.st_atime).isoformat()
    }

def load_log(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {}

def save_log(data, path):
    with open(path, 'w') as f:
        json.dump(data, f, indent=4)

def add_or_update_file(filepath, log_path, custom_metadata=None):
    """
    Adds or updates a file's metadata entry, including custom metadata.
    """
    if not os.path.isfile(filepath):
        print(f"File not found: {filepath}")
        return
    
    filename = os.path.basename(filepath)
    log = load_log(log_path)

    base_meta = get_base_metadata(filepath)
    entry = log.get(filename, {})

    # Combine existing metadata, new base, and new custom metadata
    entry.update(base_meta)
    if custom_metadata:
        entry.update(custom_metadata)

    log[filename] = entry
    save_log(log, log_path)
    print(f"Updated log for {filename}")

def read_metadata(filepath, log_path):
    """
    Adds or updates a file's metadata entry, including custom metadata.
    """
    if not os.path.isfile(filepath):
        print(f"File not found: {filepath}")
        return
    
    filename = os.path.basename(filepath)
    log = load_log(log_path)

    entry = log.get(filename, None)
    if entry:
        print(f"Metadata for {filename}:")
        for key, value in entry.items():
            print(f"{key}: {value}")
    else:
        print(f"No metadata found for {filename}")


In [15]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = os.path.join(source_path, "outputs", "preprocessed_data")
os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist

public_df = test_df[test_df['Usage']=='PublicTest']
private_df = test_df[test_df['Usage']=='PrivateTest']

output_file = os.path.join(output_dir, f"icdar_test_public_df_{timestamp}.csv")
public_df.to_csv(output_file, index=False)
print(f"Dataframe saved to {output_file}")
LOG_FILE = output_dir+"\\file_metadata_log.json"
print(f"Log file path: {LOG_FILE}")
print(f"Output file path: {output_file}")
add_or_update_file(
    output_file, LOG_FILE,
    custom_metadata={
        "seed": seed,
        "description": '''dataframe with the following columns: writer, language, same_text, isEng, train, filename, index; 
        Each row is one of the original dataset image files. I have simplified the code, previously it was unnecessarily complicated''' 
    }
)

output_file = os.path.join(output_dir, f"icdar_test_private_df_{timestamp}.csv")
private_df.to_csv(output_file, index=False)
print(f"Dataframe saved to {output_file}")
LOG_FILE = output_dir+"\\file_metadata_log.json"
print(f"Log file path: {LOG_FILE}")
print(f"Output file path: {output_file}")
add_or_update_file(
    output_file, LOG_FILE,
    custom_metadata={
        "seed": seed,
        "description": '''dataframe with the following columns: writer, language, same_text, isEng, train, filename, index; 
        Each row is one of the original dataset image files. I have simplified the code, previously it was unnecessarily complicated''' 
    }
)

Dataframe saved to c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\preprocessed_data\icdar_test_public_df_20250716_101521.csv
Log file path: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\preprocessed_data\file_metadata_log.json
Output file path: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\preprocessed_data\icdar_test_public_df_20250716_101521.csv
Updated log for icdar_test_public_df_20250716_101521.csv
Dataframe saved to c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\preprocessed_data\icdar_test_private_df_20250716_101521.csv
Log file path: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\preprocessed_data\file_metadata_log.json
Output file path: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\preprocessed_data\icdar_test_private_df_20250716_101521.csv
Updated log for icdar_test_private_df_20250716_101521.csv


In [147]:
read_metadata(
    output_file,
    log_path=LOG_FILE
)

Metadata for icdar_train_df_20250514_175905.csv:
full_path: D:\burtm\Visual_studio_code\PD_related_projects\outputs\preprocessed_data\icdar_train_df_20250514_175905.csv
size_bytes: 147733
created: 2025-05-14T17:59:05.299873
modified: 2025-05-14T17:59:05.604248
accessed: 2025-05-14T17:59:05.604248
seed: 42
description: dataframe with the following columns: writer, language, same_text, isEng, train, filename, index; 
        Each row is one of the original dataset image files. I have simplified the code, previously it was unnecessarily complicated
